# Approach 1 (avoid-step) — Averaged CE vs Fitted Curve, elbow comparison

For every (P%, BS) combination: scatter shows averaged raw CE **up to the detected step cutoff**,
overlaid with the smooth fitted curve `A + B/(BN+1)^n` where **A is the empirical tail floor**.

**Two elbow methods are shown side by side** (both computed by the compute notebook, read from
`intermediate/approach_1_fit_params_bs_{bs}.csv`):

- **Purple star — perpendicular-distance method** (primary): normalize BN and CE to [0,1] using
  the floor asymptote A; the elbow is the data point maximizing `1 − (x_norm + y_norm)`, i.e. the
  point farthest BELOW the start→(BN_end, A) diagonal. Columns: `elbow_BN`, `CE_learned`, `IPA`.
- **Orange diamond — Kneedle** (Satopää et al., 2011, `kneed` package, `curve='convex'`,
  `direction='decreasing'`, `S=1.0`): cross-check. Columns: `elbow_BN_kneedle`, `IPA_kneedle`.

The bottom-right inset shows the signed distance curve `D = 1 − (x_norm + y_norm)` that the
distance method maximizes, with both elbows marked — making the detection transparent.

PNGs saved to `BS_{bs}/fitting_avg_plot_A_1_elbow_step_p_{p}_bs_{bs}.png`.

In [ ]:
# === Cell 1 — Config, imports ===
import os, glob, re
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings("ignore")

# ── CONFIG ────────────────────────────────────────────────────────────────────────────
BN_STEP_MIN = 100
STEP_THRESH  = 0.01
KNEEDLE_S    = 1.0   # must match compute notebook
# ─────────────────────────────────────────────────────────────────────────────────────

BASE_DIR  = r"C:\Users\Student\Desktop\Projects\research\physlab\SLP\SLP-MNIST\prune_layers_ALL"
INTER_DIR = r"C:\Users\Student\Desktop\Projects\research\physlab\SLP\SLP-MNIST\IPA_methods\Approach_1\test_1\approach_1_elbow_step\intermediate"
OUT_DIR   = r"C:\Users\Student\Desktop\Projects\research\physlab\SLP\SLP-MNIST\IPA_methods\Approach_1\test_1\approach_1_elbow_step\avg_plot_v_fitting_curve_elbow"

BATCH_SIZES = [64, 1024, 60000]
CE_o = np.log(10)

p_dirs = glob.glob(os.path.join(BASE_DIR, "p-percentage_*"))
PRUNING_LEVELS = sorted([
    float(re.search(r"p-percentage_([\d.]+)", d).group(1))
    for d in p_dirs
])
print(f"Found {len(PRUNING_LEVELS)} pruning levels: {PRUNING_LEVELS}")
print(f"CE_o = ln(10) = {CE_o:.6f}")
print("Cell 1 ready.")

In [ ]:
# === Cell 2 — Load all (P%, BS) records ===
records = []   # one dict per (p, bs)

for bs in BATCH_SIZES:
    params_csv = os.path.join(INTER_DIR, f"approach_1_fit_params_bs_{bs}.csv")
    if not os.path.exists(params_csv):
        print(f"[SKIP] Missing params CSV: {params_csv}")
        continue
    params_df = pd.read_csv(params_csv)
    params_df.columns = params_df.columns.str.strip()

    for p in PRUNING_LEVELS:
        avg_csv = os.path.join(BASE_DIR, f"p-percentage_{p}", f"batch_size_{bs}",
                               f"averaged_runs_p_{p}_bs_{bs}.csv")
        if not os.path.exists(avg_csv):
            print(f"  [SKIP] P%={p*100:5.1f}%  BS={bs:>6}  — no averaged CSV")
            continue

        row = params_df[np.isclose(params_df["P%"], p * 100)]
        if row.empty:
            print(f"  [SKIP] P%={p*100:5.1f}%  BS={bs:>6}  — no fit params row")
            continue

        cutoff_BN = float(row["cutoff_BN"].iloc[0])

        full_df = pd.read_csv(avg_csv)
        full_df.columns = full_df.columns.str.strip()
        ce_col = next((c for c in full_df.columns if c in ("Avg_CE_Test", "Avg_CE_test")), None)
        bn_col = next((c for c in full_df.columns if "Batch" in c), None)
        if ce_col is None or bn_col is None:
            print(f"  [SKIP] P%={p*100:5.1f}%  BS={bs:>6}  — unexpected columns {list(full_df.columns)}")
            continue
        full_df = full_df.dropna(subset=[ce_col, bn_col])
        max_bn  = float(full_df[bn_col].max())
        step_was_detected = cutoff_BN < max_bn

        avg_df = full_df[full_df[bn_col] < cutoff_BN]
        bn_avg = avg_df[bn_col].values.astype(float)
        ce_avg = avg_df[ce_col].values.astype(float)

        A          = float(row["A"].iloc[0])
        B          = float(row["B"].iloc[0])
        n          = float(row["n"].iloc[0])
        # Distance method (primary)
        elbow_BN   = float(row["elbow_BN"].iloc[0])
        BN_learned = float(row["BN_learned"].iloc[0])
        CE_learned = float(row["CE_learned"].iloc[0])
        IPA        = float(row["IPA"].iloc[0])
        # Kneedle (cross-check)
        elbow_BN_kn   = float(row["elbow_BN_kneedle"].iloc[0])   if "elbow_BN_kneedle"   in row.columns else np.nan
        CE_learned_kn = float(row["CE_learned_kneedle"].iloc[0]) if "CE_learned_kneedle" in row.columns else np.nan
        IPA_kn        = float(row["IPA_kneedle"].iloc[0])        if "IPA_kneedle"        in row.columns else np.nan
        rmse_full  = float(row["RMSE_full"].iloc[0])   if "RMSE_full"   in row.columns else np.nan
        rmse_last50= float(row["RMSE_last50"].iloc[0]) if "RMSE_last50" in row.columns else np.nan

        from_data = np.isfinite(BN_learned) and (BN_learned <= max_bn)

        # Indices of the elbow data points
        if np.isfinite(elbow_BN) and len(bn_avg) > 0:
            elbow_idx = int(np.argmin(np.abs(bn_avg - elbow_BN)))
        else:
            elbow_idx = None
        if np.isfinite(elbow_BN_kn) and len(bn_avg) > 0:
            kneedle_idx = int(np.argmin(np.abs(bn_avg - elbow_BN_kn)))
        else:
            kneedle_idx = None

        records.append({
            "bs": bs, "p": p,
            "bn_avg":            bn_avg,
            "ce_avg":            ce_avg,
            "A":                 A,
            "B":                 B,
            "n":                 n,
            "elbow_BN":          elbow_BN,
            "elbow_idx":         elbow_idx,
            "BN_learned":        BN_learned,
            "CE_learned":        CE_learned,
            "elbow_BN_kneedle":  elbow_BN_kn,
            "kneedle_idx":       kneedle_idx,
            "CE_learned_kneedle": CE_learned_kn,
            "IPA_kneedle":       IPA_kn,
            "from_data":         from_data,
            "IPA":               IPA,
            "cutoff_BN":         cutoff_BN,
            "step_was_detected": step_was_detected,
            "RMSE_full":         rmse_full,
            "RMSE_last50":       rmse_last50,
        })
        step_tag = "[step detected]" if step_was_detected else "[no step]"
        print(f"  OK   P%={p*100:5.1f}%  BS={bs:>6}  cutoff_BN={cutoff_BN:>6.0f} {step_tag:<16}  "
              f"A(floor)={A:.4f}  elbow_BN={elbow_BN:>7.1f}  kneedle_BN={elbow_BN_kn:>7.1f}")

print(f"\nLoaded {len(records)} combinations.")

In [ ]:
# === Cell 3 — Plot averaged CE (pre-step) vs fitted curve, both elbow methods ===
#
# Each plot shows:
#   - Grey scatter:           averaged CE data (pre-step window)
#   - Blue line:              fitted curve  A + B/(BN+1)^n  (A pinned to floor)
#   - Green dashed:           A = floor asymptote
#   - Grey dotted:            CE_o reference
#   - Purple star:            elbow, perpendicular-distance method (primary)
#   - Orange diamond:         elbow, Kneedle (cross-check)
#   - Purple dashed horiz:    CE_learned  (distance method)
#   - Red vertical dashed:    BN_learned  (distance method)
#   - Middle-left inset (A):  distance view, D = 1 - (x_norm + y_norm); both elbows
#   - Bottom-right inset (B): Kneedle view, transformed curve y_t = 1 - y_hat,
#                             diagonal y = x, difference curve D_kn = y_t - x_hat

plt.rcParams.update({"font.size": 13})

for rec in records:
    bs                = rec["bs"]
    p                 = rec["p"]
    bn_avg            = rec["bn_avg"]
    ce_avg            = rec["ce_avg"]
    A                 = rec["A"]
    B                 = rec["B"]
    n                 = rec["n"]
    elbow_BN          = rec["elbow_BN"]
    elbow_idx         = rec["elbow_idx"]
    BN_learned        = rec["BN_learned"]
    CE_learned        = rec["CE_learned"]
    elbow_BN_kn       = rec["elbow_BN_kneedle"]
    kneedle_idx       = rec["kneedle_idx"]
    IPA_kn            = rec["IPA_kneedle"]
    from_data         = rec["from_data"]
    IPA               = rec["IPA"]
    cutoff_BN         = rec["cutoff_BN"]
    step_was_detected = rec["step_was_detected"]
    RMSE_full         = rec["RMSE_full"]
    RMSE_last50       = rec["RMSE_last50"]

    bn_end    = max(bn_avg.max() if len(bn_avg) > 0 else cutoff_BN,
                    BN_learned if np.isfinite(BN_learned) else 0) * 1.3
    bn_smooth = np.linspace(0, bn_end, 800)
    y_fit     = A + B / ((bn_smooth + 1) ** n)

    fig, ax = plt.subplots(figsize=(10, 6))

    # 1. Averaged CE scatter (pre-step only)
    scatter_label = (r"$\overline{CE}_{test}$ (avg 100 runs, pre-step)"
                     if step_was_detected else
                     r"$\overline{CE}_{test}$ (avg 100 runs, full data)")
    ax.scatter(bn_avg, ce_avg, s=8, color="#aaaaaa", alpha=0.6, zorder=1,
               label=scatter_label)

    # 2. Smooth fitted curve
    ax.plot(bn_smooth, y_fit, color="#1f77b4", linewidth=2.2, zorder=3,
            label=(f"Fit (A=floor): A={A:.4f},  B={B:.4f},  n={n:.4f}\n"
                   f"RMSE_last50={RMSE_last50:.4f}  (RMSE_full={RMSE_full:.4f})"))

    # 3. Step cutoff boundary line
    if step_was_detected:
        ax.axvline(cutoff_BN, color="#bbbbbb", linewidth=1.0, linestyle="-", alpha=0.6)
        ax.text(cutoff_BN + 5, CE_o - 0.05,
                f"BN={cutoff_BN:.0f}\n(step cutoff)",
                fontsize=8, color="#888888", va="top")

    # 4. CE_o reference (dotted grey)
    ax.axhline(CE_o, color="#888888", linewidth=1.0, linestyle=":")
    ax.text(bn_end, CE_o + 0.03, f"CE_o = {CE_o:.4f}",
            ha="right", fontsize=10, color="#666666")

    # 5. A = floor asymptote (green dashed)
    ax.axhline(A, color="#2ca02c", linewidth=1.4, linestyle="--")
    ax.text(bn_end, A - 0.07, f"A = {A:.4f}  (floor asymptote)",
            ha="right", fontsize=10, color="#2ca02c")

    # 6. Elbow, distance method (purple star)
    if elbow_idx is not None and elbow_idx < len(bn_avg):
        ax.scatter([bn_avg[elbow_idx]], [ce_avg[elbow_idx]], s=160, color="#9467bd",
                   marker="*", zorder=6,
                   label=f"Elbow, distance method  BN={elbow_BN:.0f}  (IPA={IPA:.5f})")

    # 6b. Elbow, Kneedle cross-check (orange diamond)
    if kneedle_idx is not None and kneedle_idx < len(bn_avg):
        ax.scatter([bn_avg[kneedle_idx]], [ce_avg[kneedle_idx]], s=90, color="#ff7f0e",
                   marker="D", zorder=6,
                   label=f"Elbow, Kneedle (S={KNEEDLE_S})  BN={elbow_BN_kn:.0f}  (IPA={IPA_kn:.5f})")

    # 7. CE_learned horizontal line (purple dashed, distance method)
    if np.isfinite(CE_learned):
        ax.axhline(CE_learned, color="#9467bd", linewidth=1.4, linestyle="--")
        ax.text(bn_end, CE_learned + 0.03, f"CE_learned = {CE_learned:.4f}",
                ha="right", fontsize=10, color="#9467bd")

    # 8. BN_learned vertical line + annotation (distance method)
    if np.isfinite(BN_learned) and BN_learned > 0:
        bnl_color = "#d62728"
        ax.axvline(BN_learned, color=bnl_color, linewidth=1.4,
                   linestyle="--", alpha=0.8, zorder=4)
        annot_y = CE_learned if np.isfinite(CE_learned) else ce_avg.mean()
        ax.annotate(
            f"elbow_BN = {elbow_BN:.0f}\nBN_learned = {BN_learned:.0f}\n"
            f"CE_learned = {CE_learned:.4f}\nIPA = {IPA:.5f}\nIPA_kneedle = {IPA_kn:.5f}",
            xy=(BN_learned, annot_y),
            xytext=(BN_learned + bn_end * 0.03, annot_y + 0.15),
            fontsize=9, color=bnl_color,
            arrowprops=dict(arrowstyle="->", color=bnl_color, lw=1.0)
        )

    ax.set_xlabel("Batch Number (BN)")
    ax.set_ylabel("CE_TEST")
    ax.set_xlim(0, bn_end)
    ax.set_ylim(max(0, A - 0.15), CE_o + 0.25)
    step_tag = f"step at BN={cutoff_BN:.0f}" if step_was_detected else "no step detected"
    ax.set_title(
        f"Approach 1 (avoid-step) — distance vs Kneedle elbow  |  P%={p*100:.1f}%  BS={bs}\n"
        f"Cutoff: {step_tag}   |   A(floor)={A:.4f}   RMSE_last50={RMSE_last50:.4f}"
    )
    ax.legend(fontsize=10, frameon=False, loc="upper right")
    ax.grid(True, alpha=0.25)

    # ── Inset A (middle-left): distance-method view, D = 1 - (x_norm + y_norm) ─────
    # Same normalization as find_elbow_distance (anchors: CE[0] and A = floor).
    BN_range_n = bn_avg[-1] - bn_avg[0] if len(bn_avg) > 1 else 0.0
    CE_range_n = ce_avg[0] - A if len(ce_avg) > 0 else 0.0

    if BN_range_n > 0 and CE_range_n > 1e-10:
        x_n = (bn_avg - bn_avg[0]) / BN_range_n
        y_n = (ce_avg - A)         / CE_range_n
        D   = 1.0 - (x_n + y_n)   # positive where the curve bows below the diagonal

        ax_d = ax.inset_axes([0.14, 0.40, 0.32, 0.28])
        ax_d.plot(bn_avg, D, color="#9467bd", linewidth=1.3, zorder=2)
        ax_d.axhline(0, color="#555555", linewidth=0.8, linestyle="--", zorder=1)
        ax_d.fill_between(bn_avg, 0, D, where=(D > 0),
                          color="#9467bd", alpha=0.15, zorder=0)
        if elbow_idx is not None and elbow_idx < len(bn_avg):
            ax_d.scatter([bn_avg[elbow_idx]], [D[elbow_idx]], s=70, color="#9467bd",
                         marker="*", zorder=5, label=f"distance  BN={elbow_BN:.0f}")
            ax_d.axvline(bn_avg[elbow_idx], color="#9467bd", linewidth=0.7,
                         linestyle=":", alpha=0.6)
        if kneedle_idx is not None and kneedle_idx < len(bn_avg):
            ax_d.scatter([bn_avg[kneedle_idx]], [D[kneedle_idx]], s=45, color="#ff7f0e",
                         marker="D", zorder=5, label=f"Kneedle  BN={elbow_BN_kn:.0f}")
        ax_d.set_xlabel("BN", fontsize=8)
        ax_d.set_ylabel("D", fontsize=8)
        ax_d.set_title(r"distance view:  $D = 1 - (\hat{x} + \hat{y})$", fontsize=8)
        ax_d.tick_params(labelsize=7)
        ax_d.grid(True, alpha=0.2)
        ax_d.set_facecolor("#f9f9f9")
        ax_d.legend(fontsize=6, frameon=False)

    # ── Inset B (bottom-right): Kneedle view — shape transform y -> 1 - y ──────────
    # Kneedle min/max normalization; the convex decreasing curve is transformed to
    # concave increasing:  y_t = 1 - y_hat.  Reference diagonal y = x.
    # Difference curve  D_kn = y_t - x_hat  peaks at the knee.
    if len(bn_avg) > 1 and np.ptp(ce_avg) > 1e-10:
        x_k = (bn_avg - bn_avg.min()) / np.ptp(bn_avg)
        y_k = (ce_avg - ce_avg.min()) / np.ptp(ce_avg)
        y_t  = 1.0 - y_k                    # Kneedle transform (convex + decreasing)
        D_kn = y_t - x_k

        ax_k = ax.inset_axes([0.60, 0.08, 0.36, 0.30])
        ax_k.plot(x_k, y_t, color="#1f77b4", linewidth=1.3, zorder=3,
                  label=r"$\hat{y}_t = 1 - \hat{y}$")
        ax_k.plot([0, 1], [0, 1], color="#555555", linewidth=0.9, linestyle="--",
                  zorder=2, label=r"$y = x$")
        ax_k.plot(x_k, D_kn, color="#ff7f0e", linewidth=1.2, zorder=2,
                  label=r"$D_{kn} = \hat{y}_t - \hat{x}$")
        if kneedle_idx is not None and kneedle_idx < len(x_k):
            ax_k.scatter([x_k[kneedle_idx]], [y_t[kneedle_idx]], s=45,
                         color="#ff7f0e", marker="D", zorder=5)
            ax_k.scatter([x_k[kneedle_idx]], [D_kn[kneedle_idx]], s=45,
                         color="#ff7f0e", marker="D", facecolors="none", zorder=5)
            ax_k.axvline(x_k[kneedle_idx], color="#ff7f0e", linewidth=0.7,
                         linestyle=":", alpha=0.7)
        if elbow_idx is not None and elbow_idx < len(x_k):
            ax_k.scatter([x_k[elbow_idx]], [y_t[elbow_idx]], s=60,
                         color="#9467bd", marker="*", zorder=5)
        ax_k.set_xlabel(r"$\hat{x}$ (BN norm.)", fontsize=8)
        ax_k.set_ylabel(r"$\hat{y}_t$,  $D_{kn}$", fontsize=8)
        ax_k.set_title("Kneedle view: transform + diagonal + D", fontsize=8)
        ax_k.tick_params(labelsize=7)
        ax_k.grid(True, alpha=0.2)
        ax_k.set_facecolor("#f9f9f9")
        ax_k.legend(fontsize=6, frameon=False, loc="upper left")
    # ─────────────────────────────────────────────────────────────────────────────────

    bs_dir  = os.path.join(OUT_DIR, f"BS_{bs}")
    os.makedirs(bs_dir, exist_ok=True)
    out_png = os.path.join(bs_dir, f"fitting_avg_plot_A_1_elbow_step_p_{p}_bs_{bs}.png")
    plt.tight_layout()
    plt.savefig(out_png, dpi=150, bbox_inches="tight")
    plt.close(fig)
    step_label = f"step@{cutoff_BN:.0f}" if step_was_detected else "no step"
    print(f"  Saved: fitting_avg_plot_A_1_elbow_step_p_{p}_bs_{bs}.png  "
          f"[{step_label}, elbow_BN={elbow_BN:.1f}, kneedle_BN={elbow_BN_kn:.1f}]")

print("\n[Done]")